this notebook contains the analysis for the incident isochrones.

# 1.0 Load library and datasets

In [ ]:
# load the packages
import pandas as pd
import json
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy.stats as stats
from shapely.geometry import Point
from matplotlib.patches import Patch
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests
from statsmodels.discrete.discrete_model import NegativeBinomial

In [ ]:
import gc
gc.collect()

In [ ]:
# load the cleaned boundary data
bradford_boundary = gpd.read_file('bradford_boundary.geojson', use_arrow=True)
bradford_imd = pd.read_csv('bradford_imd.csv')

In [ ]:
df = gpd.read_file('incident_isochrones.geojson', use_arrow=True)

The times extracted for the isochrones are 5, 10 , 15, 20, 25, and 30 minutes. Time of trips outside this are left as nan.

# 2.0 Buffer separation

In [ ]:
# filter for each time category
within_5min = df[df['trip_time_min'] <= 5].copy()
over_5min = df[(df['trip_time_min'] > 5) | (df['trip_time_min'].isna())].copy()
within_10min = df[df['trip_time_min'] <= 10].copy()
over_10min = df[(df['trip_time_min'] > 10) | (df['trip_time_min'].isna())].copy()
within_15min = df[df['trip_time_min'] <= 15].copy()
over_15min = df[(df['trip_time_min'] > 15) | (df['trip_time_min'].isna())].copy()
within_20min = df[df['trip_time_min'] <= 20].copy()
over_20min = df[(df['trip_time_min'] > 20) | (df['trip_time_min'].isna())].copy()
within_25min = df[df['trip_time_min'] <= 25].copy()
over_25min = df[(df['trip_time_min'] > 25) | (df['trip_time_min'].isna())].copy()
within_30min = df[df['trip_time_min'] <= 30].copy()
over_30min = df[(df['trip_time_min'] > 30) | (df['trip_time_min'].isna())].copy()

In [ ]:
# check that data is well filtered: replace the distances
print(over_20min['trip_time_min'].min())
print((over_20min['trip_time_min'] <= 20).sum())

In [ ]:
# tract the tables
within_5min['buffer_track'] = 'within_5'
over_5min['buffer_track'] = 'over_5'
within_10min['buffer_track'] = 'within_10'
over_10min['buffer_track'] = 'over_10'
within_15min['buffer_track'] = 'within_15'
over_15min['buffer_track'] = 'over_15'
within_20min['buffer_track'] = 'within_20'
over_20min['buffer_track'] = 'over_20'
within_25min['buffer_track'] = 'within_25'
over_25min['buffer_track'] = 'over_25'
within_30min['buffer_track'] = 'within_30'
over_30min['buffer_track'] = 'over_30'

In [ ]:
# concatenate
buffer_df = pd.concat(
    [within_5min, over_5min, within_10min, over_10min,
     within_15min, over_15min, within_20min, over_20min,
     within_25min, over_25min, within_30min, over_30min],
    ignore_index=True
)

In [ ]:
buffer_df['buffer_track'].value_counts()

In [ ]:
# extract the numerical buffer distance
buffer_df['walk_time'] = (
    buffer_df['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# identify within and further-than observations
buffer_df['location'] = (
    buffer_df['buffer_track']
    .str.split('_')
    .str[0]
    .replace({
        'within': 'Within',
        'over': 'Further than'
    })
)

# 3.0 Global Analysis

## 3.1 How many incident per week?

In [ ]:
# group each buffer by the termtime
crime_totals = (buffer_df.groupby(['buffer_track', 'year&week', 'term_time_type'])
                .size()
                .unstack(fill_value=0))

In [ ]:
# rename the columns
crime_totals = (crime_totals.rename(
    columns={'Term time': 'totalcrime_schoolday',
             'holiday': 'totalcrime_holiday'})
                .reset_index())

In [ ]:
crime_totals

## 3.2 What is the incident rate per week?

In [ ]:
# Get the number of days per term time type using the initial data, df
days_per_week = df[['year&week', 'day_name', 'term_time_type']].drop_duplicates()

In [ ]:
# get the number of days
n_termdays = (
    days_per_week[days_per_week['term_time_type'] == 'Term time']
    .groupby('year&week')
    .size()
    .rename('n_schooldays')
    .reset_index())

In [ ]:
# get the number of days
n_holidays = (
    days_per_week[days_per_week['term_time_type'] == 'holiday']
    .groupby('year&week')
    .size()
    .rename('n_holidays')
    .reset_index())

In [ ]:
# merge n_term_days and n_holidays
number_of_days = n_termdays.merge(n_holidays, on='year&week', how='outer').fillna(0)

In [ ]:
# merge the crime totals to the number of days
weekly_rate = crime_totals.merge(number_of_days, on='year&week', how='left')

In [ ]:
# calculate the weekly crime rates
weekly_rate['mean_crime_schoolday'] = weekly_rate['totalcrime_schoolday'] / weekly_rate['n_schooldays']
weekly_rate['mean_crime_holiday'] = weekly_rate['totalcrime_holiday'] / weekly_rate['n_holidays']

In [ ]:
# replace the infinities
weekly_rate[['mean_crime_schoolday', 'mean_crime_holiday']] = weekly_rate[
    ['mean_crime_schoolday', 'mean_crime_holiday']].replace([np.inf, -np.inf], np.nan)

Plot the table

In [ ]:
# set the tables
buffer_order = buffer_df['buffer_track'].unique()

In [ ]:
# set the layout
fig, axes = plt.subplots(
    nrows=8, ncols=1, figsize=(16, 40),
    sharex=False, sharey=False)

axes = axes.flatten()

# loop all the tables
for ax, buffer_name in zip(axes, buffer_order):
    # filter the buffer
    plot_data = (weekly_rate[weekly_rate['buffer_track'] == buffer_name]
        .sort_values('year&week').reset_index(drop=True))

    # use numerical positions to control the x-axis labels
    x = np.arange(len(plot_data))

    # layer the school day rate
    ax.plot(x, plot_data['mean_crime_schoolday'],
            label='Term time', marker='o', ms=3, linewidth=1.5)

    ax.plot(x, plot_data['mean_crime_holiday'],
            label='Holiday', marker='o', ms=3, linewidth=1.5)

    # label each chart
    ax.set_title(f'Crime Rate per Week at {buffer_name.replace('_',' ').title()} meters')
    ax.set_xlabel('Year and week')
    ax.set_ylabel('Mean crimes per day')
    ax.grid(alpha=0.3)
    ax.legend(loc='best')

    # show eight x-axis labels
    step = max(1, len(plot_data) // 8)
    tick_positions = x[::step]

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(plot_data['year&week'].iloc[::step],
                       rotation=0, ha='center')

# add title
fig.suptitle('Weekly crime rates by walk time', fontsize=18, y=1.01)
plt.tight_layout()
plt.show()

## 3.3 Is the term/holiday gap stable across the whole study period, or does it drift over time?

In [ ]:
# copy teh weekly data
weekly_gap = weekly_rate.copy()

# subtract the non-shoold day from schoolday
weekly_gap['rate_gap'] = (
    weekly_gap['mean_crime_schoolday']
    - weekly_gap['mean_crime_holiday']
)

In [ ]:
# split parts of teh year and week
split_parts = (
    weekly_gap['year&week']
    .str.split('-W', expand=True)
)

weekly_gap['_year'] = split_parts[0].astype(int)
weekly_gap['_week'] = split_parts[1].astype(int)

In [ ]:
# initialise the layout
fig, axes = plt.subplots(
    nrows=8, ncols=1,
    figsize=(16, 40),
    sharex=False, sharey=False)

# looop for the buffer limits
for ax, buffer_name in zip(axes, buffer_order):
    # filter for the buffer
    buffer_data = (weekly_gap[weekly_gap['buffer_track'] == buffer_name]
                   .sort_values(['_year', '_week'])
                   .reset_index(drop=True))

    # order for x
    x = range(len(buffer_data))

    # plot the school mean crime
    ax.plot(x, buffer_data['rate_gap'],
            color='darkblue', marker='o', ms=3, linewidth=1.5)

    # zero means that both rates are equal
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

    # labels and titles
    ax.set_title(f'Crime rate gap at {buffer_name.replace('_', ' ').title()} minutes')
    ax.set_xlabel('Year and week')
    ax.set_ylabel('School-day rate − holiday rate')
    ax.grid(alpha=0.3)

    # set the ticks
    step = max(1, len(buffer_data) // 10)
    tick_positions = list(
        range(0, len(buffer_data), step))

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(buffer_data['year&week'].iloc[tick_positions], rotation=0, ha='center')
fig.suptitle('School-day and holiday crime rate gap across the study period', fontsize=18, y=1.002)
plt.tight_layout()
plt.show()

## 3.4 What is teh pattern of occurence of crime types by week

In [ ]:
# count crimetypes
crime_type = (buffer_df
              .groupby(['buffer_track', 'year&week', 'incident', 'term_time_type'])
              .size()
              .unstack('term_time_type', fill_value=0)
              .reindex(columns=['Term time', 'holiday'], fill_value=0))

In [ ]:
# rename the columns
crime_type = crime_type.rename(columns={
    'Term time': 'schoolday',
    'holiday': 'holiday'})

In [ ]:
# pivot the table
crime_type = crime_type.unstack('incident', fill_value=0)

In [ ]:
# flatten the names
crime_type.columns = [
    f"n_{str(offence).strip().lower().replace('-','').replace(' ', '_')}_{term_type}"
    for term_type, offence in crime_type.columns]

# reset index
crime_type = crime_type.reset_index()

In [ ]:
crime_type_df = weekly_rate.merge(crime_type, on=['buffer_track', 'year&week'], how='left')

In [ ]:
# save the summary table
crime_type_df.to_csv('isochrone_incidenttype_weekly_summary.csv', index=False)

Merge the three tables to form a complete summary table.

In [ ]:
# confirm there is no mismatch
schoolday_cols = [c for c in crime_type_df.columns
                  if c.startswith('n_') and c.endswith('_schoolday')]

holiday_cols = [c for c in crime_type_df.columns
                if c.startswith('n_') and c.endswith('_holiday')]

In [ ]:
check_schoolday = crime_type_df[schoolday_cols].sum(axis=1)
check_holiday = crime_type_df[holiday_cols].sum(axis=1)

In [ ]:
mismatch_schoolday = crime_type_df[check_schoolday != crime_type_df['totalcrime_schoolday']]
mismatch_holiday = crime_type_df[check_holiday != crime_type_df['totalcrime_holiday']]

print("Mismatched school day rows:", len(mismatch_schoolday))
print("Mismatched holiday rows:", len(mismatch_holiday))

## 3.5 How is school day rate compared to holiday rate

In [ ]:
# get the taotal number of days
total_days = (
    df[['date_only', 'term_time_type']]
    .drop_duplicates()
    .groupby('term_time_type')
    .size())

n_schooldays = total_days['Term time']
n_holidays = total_days['holiday']

In [ ]:
buffers = crime_type_df['buffer_track'].unique()

In [ ]:
# calculate rate for each buffer
buffer_list = []

# loop for each buffer
for buffer in buffers:
    # filter the buffer
    buffer_data = crime_type_df[crime_type_df['buffer_track'] == buffer]

    # School-day rate
    schoolday_rate = buffer_data['totalcrime_schoolday'].sum() / n_schooldays
    
    # Holiday rate
    holiday_rate = buffer_data['totalcrime_holiday'].sum() / n_holidays

    # Rate ratio and percentage difference
    rate_ratio = schoolday_rate / holiday_rate
    percent_difference = (
        (schoolday_rate - holiday_rate)
        / holiday_rate
    ) * 100

    # append the rates
    buffer_list.append({
        'buffer_track': buffer,
        'schoolday_rate': schoolday_rate,
        'holiday_rate': holiday_rate,
        'rate_ratio': rate_ratio,
        'percent_difference': percent_difference
    })

In [ ]:
# put into a table
rate_comparison = pd.DataFrame(buffer_list)
rate_comparison

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[buffer_order, ['schoolday_rate', 'holiday_rate']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 6), width=0.8)

# add labels and titles
ax.set_title('School day and holiday crime rates by buffer', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
within_order = ['within_5', 'within_10', 'within_15',
                'within_20', 'within_25', 'within_30']
over_order = ['over_5', 'over_10', 'over_15',
              'over_20', 'over_25', 'over_30']

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[within_order, ['schoolday_rate', 'holiday_rate']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Within Walk Time Comparison: School day and holiday crime rates by buffer', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plot_data = (
    rate_comparison
    .set_index('buffer_track')
    .loc[within_order, ['schoolday_rate', 'holiday_rate']]
)

In [ ]:
rate_change = (
    rate_comparison
    .set_index('buffer_track')
    .loc[within_order, ['schoolday_rate', 'holiday_rate']]
    .pct_change()
    .mul(100)
    .dropna()
)

In [ ]:
ax = rate_change.plot(
    kind='line',
    marker='o',
    figsize=(10, 5),
    linewidth=2
)

ax.axhline(0, color='black', linestyle='--', linewidth=1)

ax.set_title(
    'Percentage Change in Incident Rate as the Buffer Expands',
    fontweight='bold'
)
ax.set_xlabel('Buffer')
ax.set_ylabel('Change from previous buffer (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(['School day', 'Holiday'])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[within_order, ['percent_difference']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Within Walk Time Percent Comparison', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Incident Rate (incidents per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[over_order, ['schoolday_rate', 'holiday_rate']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Beyond Walk Time Comparison: School day and holiday crime rates by buffer', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Incident Rate (incidents per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# get the within only
plot_data = (
    rate_comparison
    .set_index('buffer_track')
    .loc[over_order, ['schoolday_rate', 'holiday_rate']]
)

In [ ]:
# calculate chage rate
rate_change = (
    rate_comparison
    .set_index('buffer_track')
    .loc[over_order, ['schoolday_rate', 'holiday_rate']]
    .pct_change()
    .mul(100)
    .dropna()
)

In [ ]:
# plot rate of change
ax = rate_change.plot(
    kind='line',
    marker='o',
    figsize=(10, 5),
    linewidth=2
)

# add baseline
ax.axhline(0, color='black', linestyle='--', linewidth=1)

ax.set_title(
    'Percentage Change in Incident Rate Outside the Buffer as the Buffer Expands',
    fontweight='bold'
)
ax.set_xlabel('Buffer')
ax.set_ylabel('Change from previous buffer (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(['School day', 'Holiday'])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[over_order, ['percent_difference']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Within Buffer Percent Comparison', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Incident Rate (incidents per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.6 Is crime effect more concentrated wthin the distance or furthan than the distance?

In [ ]:
# extract the numerical buffer distance
rate_comparison['walk_time'] = (
    rate_comparison['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# identify whether the observation is within or further than
rate_comparison['location'] = (
    rate_comparison['buffer_track']
    .str.split('_')
    .str[0]
    .replace({
        'within': 'Within',
        'over': 'Further than'
    })
)

In [ ]:
# pivot the table
w2f_rate_df = (
    rate_comparison
    .pivot(index='walk_time',
           columns='location',
           values=['schoolday_rate', 'holiday_rate']))

In [ ]:
# flatten and rename column
w2f_rate_df.columns = [
    f'{location}_{term}'
    for term, location in w2f_rate_df.columns]

w2f_rate_df = w2f_rate_df.reset_index()

In [ ]:
# reorder table
w2f_rate_df = w2f_rate_df[
    ['walk_time','Within_schoolday_rate','Within_holiday_rate',
     'Further than_schoolday_rate', 'Further than_holiday_rate']]

In [ ]:
w2f_rate_df['schoolday_w2f_ratio'] = (
    w2f_rate_df['Within_schoolday_rate']
    / w2f_rate_df['Further than_schoolday_rate']
)

w2f_rate_df['holiday_w2f_ratio'] = (
    w2f_rate_df['Within_holiday_rate']
    / w2f_rate_df['Further than_holiday_rate']
)

In [ ]:
w2f_rate_df

W2F ratios increased from approximately 0.07 at 5 minutes to 1.26 at 30 minutes. This indicates that crime was more prevalent outside the smallest school catchments but became increasingly concentrated within larger school-accessible areas, with the transition occurring at approximately the 20-minute catchment. School-day and holiday patterns were broadly similar.

## 3.7 For each distance comparison, do the observed school-day and holiday crime counts within and beyond the buffer differ from the counts expected based on the amount of school-day and holiday exposure?

In [ ]:
# Exposure days
total_days = n_schooldays + n_holidays

schoolday_prop = n_schooldays / total_days
holiday_prop = n_holidays / total_days

In [ ]:
# set buffers
buffer_pairs = {
    5: ('within_5', 'over_5'),
    10: ('within_10', 'over_10'),
    15: ('within_15', 'over_15'),
    20: ('within_20', 'over_20'),
    25: ('within_25', 'over_25'),
    30: ('within_30', 'over_30'),
}

In [ ]:
crime_totals.columns

In [ ]:
# aggregate from weekly crime counts
buffer_counts = (
    crime_totals
    .groupby('buffer_track')[
        ['totalcrime_schoolday', 'totalcrime_holiday']
    ]
    .sum()
).reset_index()

In [ ]:
# Extract walking time
buffer_counts['walk_time'] = (
    buffer_counts['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# Extract within or over
buffer_counts['location'] = (
    buffer_counts['buffer_track']
    .str.split('_')
    .str[0]
)

In [ ]:
# unsatck the loaction
gof_table = (
    buffer_counts
    .set_index(['walk_time', 'location'])
    [['totalcrime_schoolday', 'totalcrime_holiday']]
    .unstack('location')
)

In [ ]:
# arrange and rename the columns
gof_table = gof_table[[
    ('totalcrime_schoolday', 'within'),
    ('totalcrime_holiday', 'within'),
    ('totalcrime_schoolday', 'over'),
    ('totalcrime_holiday', 'over')
]]

gof_table.columns = [
    'Within school day',
    'Within holiday',
    'Further school day',
    'Further holiday'
]

gof_table = (
    gof_table
    .reset_index()
    .sort_values('walk_time')
)

In [ ]:
# apply the goodness of fit
gof_results = []

for _, row in gof_table.iterrows():

    # Observed counts
    observed = [
        row['Within school day'],
        row['Within holiday'],
        row['Further school day'],
        row['Further holiday']]

    # Total crimes in each location
    within_total = (row['Within school day'] + row['Within holiday'])
    further_total = (row['Further school day'] + row['Further holiday'])

    # Expected counts based on exposure days
    expected = [
        within_total * schoolday_prop,
        within_total * holiday_prop,
        further_total * schoolday_prop,
        further_total * holiday_prop
    ]

    # Goodness-of-fit test
    chi2, p_value = stats.chisquare(
        f_obs=observed,
        f_exp=expected,
        ddof=1
    )

    gof_results.append({
        'walk_time': row['walk_time'],
        'chi_square': chi2,
        'p_value': p_value
    })

In [ ]:
buffer_test = pd.DataFrame(gof_results)
buffer_test

## 3.8 Are crimes disproportionately concentrated on school days within and beyond the school buffer areas?

In [ ]:
gof_table['Within total'] = (
    gof_table['Within school day']
    + gof_table['Within holiday']
)

gof_table['Further total'] = (
    gof_table['Further school day']
    + gof_table['Further holiday']
)

In [ ]:
# calculate expected count
gof_table['Expected within school day'] = (
    gof_table['Within total'] * schoolday_prop)

gof_table['Expected within holiday'] = (
    gof_table['Within total'] * holiday_prop)

gof_table['Expected further school day'] = (
    gof_table['Further total'] * schoolday_prop)

gof_table['Expected further holiday'] = (
    gof_table['Further total'] * holiday_prop)

In [ ]:
# compare observed with expected
gof_table['Within difference (%)'] = (
    (gof_table['Within school day'] - gof_table['Expected within school day'])
    / gof_table['Expected within school day']) * 100

gof_table['Further difference (%)'] = (
    (gof_table['Further school day'] - gof_table['Expected further school day'])
    / gof_table['Expected further school day']) * 100

In [ ]:
# view table
gof_table[[
    'walk_time', 'Within school day', 'Expected within school day',
    'Within difference (%)', 'Further school day', 'Expected further school day',
    'Further difference (%)'
]].round(2)

Observed crime counts exceeded expected crime counts across all travel-time catchments and areas beyond them. Within the catchments, the percentage difference declined steadily from 3.76% at 5 minutes to 2.19% at 30 minutes, indicating that the excess of observed over expected crime was greatest in areas closest to schools. Beyond the catchments, observed counts were also consistently higher than expected, although the differences were smaller and remained relatively stable, ranging from 1.73% to 1.99%. Overall, the results suggest that the largest deviation between observed and expected crime counts occurs within the most accessible areas surrounding schools, with the magnitude of this deviation decreasing as travel-time catchments expand.

# 4.0 Incident types analysis

## 4.1 How do the incident types share the composition?

In [ ]:
# aggregate crime by types
crime_composition = (buffer_df
                     .groupby(['buffer_track', 'term_time_type', 'incident'])
                     .size()
                     .rename('crime_count')
                     .reset_index())

In [ ]:
# compute teh total crime
crime_composition['total_crimes'] = (
    crime_composition
    .groupby(['buffer_track', 'term_time_type'])['crime_count']
    .transform('sum'))

In [ ]:
# calculate share
crime_composition['share_percent'] = (
    crime_composition['crime_count']
    / crime_composition['total_crimes']) * 100

In [ ]:
# pivot the table
composition_table = (
    crime_composition
    .pivot(index=['buffer_track', 'incident'],
           columns='term_time_type',
           values='share_percent')
    .fillna(0)
    # rename columns
    .rename(columns={
        'Term time': 'schoolday_share',
        'holiday': 'holiday_share'})
    .reset_index())

In [ ]:
# calculate difference in percentage points
composition_table['share_difference_pp'] = (
    composition_table['schoolday_share']
    - composition_table['holiday_share']
)

In [ ]:
composition_table

In [ ]:
# get the offences
offences = sorted(composition_table['incident'].unique())

In [ ]:
# set teh colors
colours = (list(plt.cm.tab20.colors)
           + list(plt.cm.tab20b.colors))

colours = colours[:len(offences)]

In [ ]:
fig, axes = plt.subplots(
    nrows=8, ncols=1,
    figsize=(15, 34), sharex=False)

# loop for buffer order and transpose
for ax, buffer_name in zip(axes, buffer_order):
    plot_data = (
        composition_table[composition_table['buffer_track'] == buffer_name]
        .set_index('incident')
        .reindex(offences)
        [['schoolday_share', 'holiday_share']]
        .fillna(0)
        .T)
    
    plot_data.index = ['School day', 'Holiday']
    plot_data.plot(kind='barh', stacked=True, ax=ax,
                   color=colours, width=0.65, legend=False)

    # labels and titles
    ax.set_title(f'Share of Incident types at {buffer_name.replace('_', ' ').title()} minutes')
    ax.set_xlim(0, 100)
    ax.set_ylabel('')
    ax.set_xlabel('Share of crimes (%)')
    ax.grid(axis='x', alpha=0.3)

# set one shared leged for the crimetypes
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Crime type',
           loc='center left', bbox_to_anchor=(0.81, 0.5))

fig.suptitle('Incident-type composition on school days and holidays', fontsize=18, y=0.995)
plt.tight_layout(rect=[0, 0, 0.80, 0.98])
plt.show()

## 4.2 What is the rate of occurence of each type within the period?

In [ ]:
# count each offence by buffer and term type
offence_rate = (
    buffer_df
    .groupby(['buffer_track', 'incident', 'term_time_type'])
    .size()
    .unstack('term_time_type', fill_value=0)
    .reindex(columns=['Term time', 'holiday'], fill_value=0)
    
    # rename columns
    .rename(columns={'Term time': 'schoolday_count',
                     'holiday': 'holiday_count'})
    .reset_index())

In [ ]:
# calculate school day rate
offence_rate['schoolday_rate'] = (offence_rate['schoolday_count']
                                  / n_schooldays)

# calculate holiday rate
offence_rate['holiday_rate'] = (offence_rate['holiday_count'] / n_holidays)

In [ ]:
# calculate teh rate ratio
offence_rate['rate_ratio'] = (
    offence_rate['schoolday_rate']
    / offence_rate['holiday_rate'])

In [ ]:
# calculate the percentage difference
offence_rate['percent_difference'] = (
    offence_rate['rate_ratio'] - 1) * 100

In [ ]:
pd.set_option('display.max_row', None)
offence_rate[['buffer_track', 'incident', 'schoolday_count', 'holiday_count',
              'schoolday_rate', 'holiday_rate', 'rate_ratio', 'percent_difference']]

In [ ]:
# sort offences
offence_order = (
    offence_rate
    .groupby('incident')[['schoolday_rate', 'holiday_rate']]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=False)
    .index)

In [ ]:
# create seaparte tables for each type
# create for school day rate
crimetype_schoolday_rate = (
    offence_rate
    .pivot(index='incident',
           columns='buffer_track',
           values='schoolday_rate')
    .reindex(index=offence_order, columns=buffer_order))

# create for holiday rate
crimetype_holiday_rate = (
    offence_rate
    .pivot(
        index='incident',
        columns='buffer_track',
        values='holiday_rate')
    .reindex(index=offence_order,
             columns=buffer_order))

In [ ]:
# set maximum for both charts
maximum_rate = max(
    crimetype_schoolday_rate.max().max(),
    crimetype_holiday_rate.max().max())

In [ ]:
# set teh layout
fig, axes = plt.subplots(nrows=1, ncols=2,
                         figsize=(20, 25), sharey=True)

# plot for the school day rate
sns.heatmap(crimetype_schoolday_rate, ax=axes[0],
            cmap='YlOrRd', vmin=0, vmax=maximum_rate, cbar=False)

# plot the holiday map
sns.heatmap(crimetype_holiday_rate, ax=axes[1], cmap='YlOrRd',
            vmin=0, vmax=maximum_rate, cbar_kws={'label': 'Rate (crimes per day)'})

# set labels and titles
axes[0].set_title('School day occurrence rates', fontweight='bold')
axes[1].set_title('Holiday occurrence rates', fontweight='bold')
axes[0].set_xlabel('Buffer')
axes[1].set_xlabel('Buffer')
axes[0].set_ylabel('')
axes[1].set_ylabel('')

axes[0].tick_params(axis='x', rotation=0)
axes[1].tick_params(axis='x', rotation=0)

fig.suptitle('Crime-type occurrence rates by buffer and term type', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# sum the counts
offence_rate['total'] = offence_rate['schoolday_count'] + offence_rate['holiday_count']

In [ ]:
offence_rate

In [ ]:
# Crime types with more than 50 crimes in over_250
interest = (
    offence_rate.loc[
        (offence_rate['buffer_track'] == 'over_5')
        & (offence_rate['total'] > 50),
        'incident'
    ]
    .unique()
)

In [ ]:
# Select these crime types across all buffers
crimetype_rate_ratio = (
    offence_rate.loc[
        offence_rate['incident'].isin(interest),
        ['buffer_track', 'incident', 'rate_ratio']
    ]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['rate_ratio'])
    .copy()
)

In [ ]:
# pivot the table
df_crimetype_rate_ratio = (
    crimetype_rate_ratio
    .pivot_table(index='incident',
                 columns='buffer_track',
                 values='rate_ratio',
                 aggfunc='first')
    .reindex(index=offence_order,
             columns=buffer_order).dropna()
)

In [ ]:
df_crimetype_rate_ratio

In [ ]:
# plot the rate ratio
plt.figure(figsize=(14, 25))
sns.heatmap(
    df_crimetype_rate_ratio,
    mask=df_crimetype_rate_ratio.isna(),
    cmap='YlOrRd',
    center=1,
    annot=True,
    fmt='.2f'
)

plt.ylabel('')
plt.xlabel('')
plt.show()

In [ ]:
# set the buffers
times = [5, 10, 15, 20, 25, 30]

In [ ]:
# initialise layout
fig, axes = plt.subplots(
    nrows=4, ncols=1,
    figsize=(15, 60), sharex=False, sharey=False)

# loop for each
for ax, minute in zip(axes, times):
    # Names used in buffer_track
    within_name = f'within_{minute}'
    over_name = f'over_{minute}'

    # Select the within and over data for this distance
    chart_data = (
        crimetype_rate_ratio[crimetype_rate_ratio['buffer_track'].isin([within_name, over_name])]
        .pivot_table(index='incident',
                     columns='buffer_track',
                     values='rate_ratio',
                     aggfunc='first')
        .reindex(columns=[within_name, over_name])
        .rename(columns={within_name: 'Within',
                         over_name: 'Over'})
        .dropna(how='all'))

    # Sort by the Within rate ratio
    chart_data = chart_data.sort_values(by='Within', ascending=True,na_position='first')

    # Plot horizontal bars
    chart_data.plot(kind='barh', ax=ax, width=0.8)

    # A ratio of 1 means equal school-day and holiday rates
    ax.axvline(x=1,color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{minute} minutes walk time: crime-type rate ratios')
    ax.set_xlabel('School-day-to-holiday rate ratio')
    ax.set_ylabel('Crime type')

    ax.legend(title='Buffer category',loc='best')
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Crime-type rate ratios within and beyond each walk time', fontsize=18, y=1.002)
plt.tight_layout()
plt.show()

## 4.3 How does the within crime differ from the further crimes?

In [ ]:
# copy the offence table
comparison_table = offence_rate.copy()
# comparison_table

In [ ]:
# clean the distance track column
# extract the metric measure
comparison_table['walk_time'] = (
    comparison_table['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int))

In [ ]:
# split to create within or outside category and rename the splits
comparison_table['location'] = (
    comparison_table['buffer_track']
    .str.split('_')
    .str[0]                  # use the first part of the split
    # map the names
    .map({'within': 'Within',
          'over': 'Further than'
    }))

In [ ]:
# pivot the table
comparison_table = comparison_table.pivot(
    index=['walk_time', 'incident'],
    columns='location',
    values=['schoolday_count', 'holiday_count', 'schoolday_rate', 'holiday_rate', 'rate_ratio']
)

In [ ]:
# reorder multiindex levels
comparison_table.columns = (
    comparison_table.columns.reorder_levels([1, 0])
)

In [ ]:
# arrange Within first, followed by Further than
comparison_table = comparison_table.reindex(
    columns=['Within', 'Further than'],
    level=0
)

In [ ]:
# return distance and offence to ordinary columns
comparison_table = comparison_table.reset_index()

In [ ]:
# flatten and rename the columns
comparison_table.columns = [
    'walk_time',
    'Crime type',
    'Within: term count',
    'Within: holiday count',
    'Within: term rate',
    'Within: holiday rate',
    'Within: rate ratio',
    'Further than: term count',
    'Further than: holiday count',
    'Further than: term rate',
    'Further than: holiday rate',
    'Further than: rate ratio'
]

In [ ]:
# compare the schoold day rate within buffer to its beyond counterpart
comparison_table['schoolday w2f ratio'] = (comparison_table['Within: term rate']
                                           / comparison_table['Further than: term rate'])

comparison_table['holiday w2f ratio'] = (comparison_table['Within: holiday rate']
                                         / comparison_table['Further than: holiday rate'])

In [ ]:
# add checks to the number of crimes
comparison_table['Total within'] = (comparison_table['Within: term count']
                                    + comparison_table['Within: holiday count'])

comparison_table['Total further'] = (comparison_table['Further than: term count']
                                     + comparison_table['Further than: holiday count'])

comparison_table['Total'] = (comparison_table['Total within']
                             + comparison_table['Total further'])

In [ ]:
# set the layout
fig, axes = plt.subplots(
    nrows=4, ncols=1,
    figsize=(12, 60), sharex=False)

# loop for each wallk time
for minute, ax in zip(times, axes):
    plot_data = (
        comparison_table[comparison_table['walk_time'] == minute]
        .set_index('Crime type')[['schoolday w2f ratio', 'holiday w2f ratio']]
        .sort_values('schoolday w2f ratio', ascending=True))

    # plot the data
    plot_data.plot(kind='barh', ax=ax, width=0.8)

    # add baseline
    ax.axvline(x=1, color='black', linestyle='--', linewidth=1)

    # labels and titles
    ax.set_title(f'{minute} minutes: Within-to-Further rate ratio')
    ax.set_xlabel('Within-to-Further rate ratio')
    ax.set_ylabel('Crime type')
    ax.legend(title='Day type')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4.4 Does the school-day-to-holiday rate ratio change as the buffer increases?

In [ ]:
#get the unique offences
unique_offences = comparison_table['Crime type'].unique()

In [ ]:
# loop through each offence
for offence in unique_offences:

    offence_data = comparison_table[comparison_table['Crime type'] == offence]
    offence_data = offence_data.sort_values('walk_time')

    plt.figure(figsize=(15, 4.5))

    # within rate ratio
    plt.plot(offence_data['walk_time'], offence_data['Within: rate ratio'],
             marker='o', linewidth=2, color='#1f77b4', label='Within')

    # further-than rate ratio
    plt.plot(offence_data['walk_time'], offence_data['Further than: rate ratio'],
             marker='s', linewidth=2, color='#ff7f0e', linestyle='--', label='Further than')

    # reference value
    plt.axhline(y=1, color='black', linestyle=':', linewidth=1)

    plt.title(f'Decay Profile for {offence.title()}', fontsize=12, fontweight='bold')

    plt.xlabel('Walk_time from school (min)')
    plt.ylabel('School-day-to-holiday rate ratio')
    plt.xticks(offence_data['walk_time'])
    plt.grid(True, linestyle=':', alpha=0.4)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()

## 4.5 By what percentage did the rate ratio change when moving from one buffer distance to the next?

In [ ]:
# loop through crime types
for offence in unique_offences:

    offence_data = (comparison_table[comparison_table['Crime type'] == offence]
                    .sort_values('walk_time')
                    .copy())

    # percentage change from the previous distance
    offence_data['within_rolling_pct'] = (offence_data['Within: rate ratio']
                                          .pct_change(fill_method=None) * 100)

    offence_data['further_rolling_pct'] = (offence_data['Further than: rate ratio']
                                           .pct_change(fill_method=None) * 100)

    # set the layout
    plt.figure(figsize=(12, 4.5))

    # plot the within
    plt.plot(offence_data['walk_time'].iloc[1:], offence_data['within_rolling_pct'].iloc[1:],
             marker='o', linewidth=2, label='Within')

    plt.plot(offence_data['walk_time'].iloc[1:], offence_data['further_rolling_pct'].iloc[1:],
             marker='s', linewidth=2, linestyle='--', label='Further than')

    plt.axhline(y=0, color='black', linestyle=':', alpha=0.5)

    plt.title(f'Percentage Change by Distance: {offence.title()}', fontweight='bold')
    plt.xlabel('Walk Time (min)')
    plt.ylabel('Change from previous walk time (%)')
    plt.xticks(offence_data['walk_time'].iloc[1:])
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

## 4.6 Which of the crime types is associated school day?

In [ ]:
# calculate the ratio of what is happening within at school time to what is happening at holiday
comparison_table['w2f ratio of ratio'] = (comparison_table['schoolday w2f ratio']
                                          / comparison_table['holiday w2f ratio'])

In [ ]:
comparison_table = comparison_table.replace([np.inf, -np.inf], np.nan)

In [ ]:
times = comparison_table['walk_time'].unique()
times

In [ ]:
# initialise layout
fig, axes = plt.subplots(
    nrows=6, ncols=1,
    figsize=(16, 60), sharex=False)

# loop for each distance
for minute, ax in zip(times, axes):
    plot_data = (
        comparison_table[comparison_table['walk_time'] == minute]
        .set_index('Crime type')['w2f ratio of ratio'])

    # exclude inf and sort
    plot_data = plot_data[np.isfinite(plot_data)].sort_values(ascending=True)

    # plot the data
    plot_data.plot(kind='barh', ax=ax, color='steelblue')

    ax.axvline(x=1, color='black', linestyle='--', linewidth=1)

    # labels and titles
    ax.set_title(f'{minute} m: School-day association by crime type')
    ax.set_xlabel('Ratio of W2F ratios')
    ax.set_ylabel('Crime type')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4.7 Do the observed school-day and holiday crime counts differ from the counts expected based on the number of exposure days?

In [ ]:
# select the variables
crimes_test_df = comparison_table[
    ['walk_time', 'Crime type', 'Within: term count',
     'Within: holiday count', 'Further than: term count',
     'Further than: holiday count', 'Total']].copy()

In [ ]:
# calculate  total number of days
total_days = n_schooldays + n_holidays

# calculate proportions of each day type
schoolday_prop = n_schooldays / total_days
holiday_prop = n_holidays / total_days

In [ ]:
# calculate goodness of fit
results = []

# iterate the rows
for _, row in crimes_test_df.iterrows():
    # get observed crimes
    schoolday_within = row['Within: term count']
    holiday_within  = row['Within: holiday count']
    schoolday_further = row['Further than: term count']
    holiday_further  = row['Further than: holiday count']
    
    # calculate total volumes per spatial zone to protect against geographic area size differences
    total_within = schoolday_within + holiday_within
    total_further = schoolday_further + holiday_further
    
    # calculate expected counts
    schoolday_within_exp = total_within * schoolday_prop
    holiday_within_exp  = total_within * holiday_prop
    schoolday_further_exp = total_further * schoolday_prop
    holiday_further_exp  = total_further * holiday_prop

    # compute the ch12 test
    observed_cells = [schoolday_within, holiday_within, schoolday_further, holiday_further]
    expected_cells = [schoolday_within_exp, holiday_within_exp, schoolday_further_exp, holiday_further_exp]
    
    # run adjusted Chi-Square test
    # ddof=1 adjusts the degrees of freedom to exactly 2, since 2 localised totals are used
    chi2_stat, p_val = stats.chisquare(f_obs=observed_cells, f_exp=expected_cells, ddof=1)

    # append the values
    results.append({
        'Crime type': row['Crime type'],
        'walk_time': row['walk_time'],
        'Chi-square': chi2_stat,
        'p_value': p_val
    })

In [ ]:
crimes_chi_results = pd.DataFrame(results)

In [ ]:
# merge with teh selected group
crimes_test_df = crimes_test_df.merge(
    crimes_chi_results, 
    on=['Crime type', 'walk_time'], 
    how='left'
)

In [ ]:
# sort the p values
crimes_test_df = crimes_test_df.sort_values(by='p_value', ascending=True)
crimes_test_df[crimes_test_df['p_value'] < 0.05]

In [ ]:
# convert the results list into another DataFrame
results_df = pd.DataFrame(results)

In [ ]:
# ensure p-values are numeric
results_df['p_value'] = pd.to_numeric(
    results_df['p_value'],
    errors='coerce'
)

In [ ]:
# identify valid p-values
valid_p = (results_df['p_value'].notna()
           & np.isfinite(results_df['p_value']))

adjust the p-value

In [ ]:
# create an empty adjusted column
results_df['adjusted_p_value'] = np.nan

# adjust only valid p-values
results_df.loc[valid_p, 'adjusted_p_value'] = multipletests(
    results_df.loc[valid_p, 'p_value'], method='fdr_bh')[1]

In [ ]:
adjusted = results_df[results_df['adjusted_p_value'] < 0.05]
adjusted['Crime type'].unique()

after adjustment, only th ethe public order offences and violence without injury are associated accross all walk time.